# Single Node Model Training Example

This notebook demonstrates how to train a model on a single node, extracted from the federated learning codebase. You can easily modify the model and dataset configurations to experiment with different setups.


In [1]:
# Import required libraries
import sys
import os
import torch
import torch.nn as nn
from torch import optim
from tqdm import tqdm
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Add the project root to the path
sys.path.insert(0, os.path.abspath('.'))

# Import project utilities
from app.config import config
from app.util import model_utils, data_utils
from app.model.utils import get_available_torch_device

# Set random seeds for reproducibility
import numpy as np
np.random.seed(0)
torch.manual_seed(0)
if torch.cuda.is_available():
    torch.cuda.manual_seed(0)


## Configuration

Modify these parameters to experiment with different models and datasets:

**Note on GPU Usage:**
- The notebook automatically detects and uses GPU if available
- If you have multiple GPUs and want to use a specific one, you can manually set `device = torch.device('cuda:0')` or `torch.device('cuda:1')`
- To force CPU usage, set `device = torch.device('cpu')`


In [3]:
# Configuration - Modify these to change model and dataset
config.model_name = 'VGG'  # Options: 'VGG', 'VGG16'
config.dataset_name = 'cifar10'  # Options: 'cifar10', 'cifar100'
config.dataset_path = 'app/dataset/data/'

# Training hyperparameters
LEARNING_RATE = 0.01
BATCH_SIZE = 100
NUM_EPOCHS = 10
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
LR_STEP_SIZE = 20
LR_GAMMA = 0.1

# Device configuration
# Automatically detects GPU if available, otherwise uses CPU
device = get_available_torch_device()

# GPU-specific settings
USE_GPU = torch.cuda.is_available()
PIN_MEMORY = USE_GPU  # Pin memory for faster GPU transfer

# Display device information
print("=" * 50)
print("Device Configuration:")
print(f"  CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"  CUDA Version: {torch.version.cuda}")
print(f"  Using Device: {device}")
print(f"  Pin Memory: {PIN_MEMORY}")
print("=" * 50)


Device Configuration:
  CUDA Available: True
  GPU Device: NVIDIA H100 NVL
  GPU Memory: 93.09 GB
  CUDA Version: 12.1
  Using Device: cuda
  Pin Memory: True


## Load Dataset


In [11]:
# Load training dataset
trainset = data_utils.get_trainset()
trainloader = DataLoader(
    trainset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=2,
    pin_memory=PIN_MEMORY  # Faster GPU transfer when using GPU
)

# Load test dataset
testset = data_utils.get_testset()
testloader = DataLoader(
    testset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=2,
    pin_memory=PIN_MEMORY  # Faster GPU transfer when using GPU
)

print(f"Training samples: {len(trainset)}")
print(f"Test samples: {len(testset)}")
print(f"Number of classes: {len(trainset.classes) if hasattr(trainset, 'classes') else 'Unknown'}")


100%|██████████████████████████████████████| 170498071/170498071 [00:13<00:00, 12193937.65it/s]


Extracting app/dataset/data/cifar10/cifar10.tar.gz to app/dataset/data/cifar10/
Training samples: 50000
Test samples: 10000
Number of classes: 10


## Initialize Model


In [6]:
# Create model - 'Unit' means full model (not split)
# None means no layer splitting
# False means not edge-based
net = model_utils.get_model('Unit', None, device, False)
net = net.to(device)

# Print model architecture
print("Model Architecture:")
print(net)
print(f"\nTotal parameters: {sum(p.numel() for p in net.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in net.parameters() if p.requires_grad):,}")


Model Architecture:
VGG(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
  )
  (denses): Sequential(
    (0): Linear(in_features=4096, out_features=128, bias=True)
    (1): Linear(in_features=128, out_features=10, bias=True)
  )
)

Total parameters: 582,346
Trainable parameters

## Setup Optimizer, Loss Function, and Scheduler


In [7]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer (SGD with momentum)
optimizer = optim.SGD(
    net.parameters(), 
    lr=LEARNING_RATE, 
    momentum=MOMENTUM, 
    weight_decay=WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(
    optimizer, 
    step_size=LR_STEP_SIZE, 
    gamma=LR_GAMMA
)

print(f"Optimizer: SGD(lr={LEARNING_RATE}, momentum={MOMENTUM}, weight_decay={WEIGHT_DECAY})")
print(f"Scheduler: StepLR(step_size={LR_STEP_SIZE}, gamma={LR_GAMMA})")


Optimizer: SGD(lr=0.01, momentum=0.9, weight_decay=0.0005)
Scheduler: StepLR(step_size=20, gamma=0.1)


## Training Loop

This mimics the training code from `fed_client.py` - the no-offloading training case.


In [8]:
# Training loop
net.train()
train_losses = []
train_accuracies = []

for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    correct = 0
    total = 0
    
    # Training phase
    pbar = tqdm(trainloader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    for batch_idx, (inputs, targets) in enumerate(pbar):
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = net(inputs)
        loss = criterion(outputs, targets)
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f'{running_loss/(batch_idx+1):.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })
    
    # Update learning rate
    scheduler.step()
    
    # Calculate epoch statistics
    epoch_loss = running_loss / len(trainloader)
    epoch_acc = 100. * correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)
    
    print(f'Epoch {epoch+1}: Loss={epoch_loss:.4f}, Accuracy={epoch_acc:.2f}%, LR={scheduler.get_last_lr()[0]:.6f}')
    
    # Test accuracy after each epoch
    net.eval()
    test_loss = 0
    test_correct = 0
    test_total = 0
    
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = net(inputs)
            loss = criterion(outputs, targets)
            
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            test_total += targets.size(0)
            test_correct += predicted.eq(targets).sum().item()
    
    test_acc = 100. * test_correct / test_total
    print(f'Test Accuracy: {test_acc:.2f}%\n')
    
    net.train()


Epoch 1/10: 100%|██████████████████████| 500/500 [00:09<00:00, 54.69it/s, loss=1.5274, acc=44.20%]

Epoch 1: Loss=1.5274, Accuracy=44.20%, LR=0.010000


Test Accuracy: 58.58%



Epoch 2/10: 100%|██████████████████████| 500/500 [00:09<00:00, 53.59it/s, loss=1.1708, acc=58.01%]

Epoch 2: Loss=1.1708, Accuracy=58.01%, LR=0.010000


...

Epoch 9: Loss=0.7540, Accuracy=73.48%, LR=0.010000


Test Accuracy: 74.93%



Epoch 10/10: 100%|█████████████████████| 500/500 [00:08<00:00, 58.74it/s, loss=0.7238, acc=74.68%]

Epoch 10: Loss=0.7238, Accuracy=74.68%, LR=0.010000


Test Accuracy: 74.92%



## Final Evaluation

Using the test function from `model_utils.py`


In [9]:
# Final test evaluation
final_accuracy = model_utils.test(net, testloader, device, criterion)
print(f"\nFinal Test Accuracy: {final_accuracy:.2f}%")


100%|███████████████████████████████████████████████████████████| 100/100 [00:01<00:00, 85.65it/s]
model_utils.py:133 - 2025-11-23 13:03:06,920 - INFO - Test Accuracy: 74.92



Final Test Accuracy: 74.92%


## Save Model

The model is automatically saved by `model_utils.test()`, but you can also save it manually:


In [10]:
# Save model checkpoint
model_path = f'./{config.model_name}_{config.dataset_name}_trained.pth'
torch.save({
    'epoch': NUM_EPOCHS,
    'model_state_dict': net.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'accuracy': final_accuracy,
    'config': {
        'model_name': config.model_name,
        'dataset_name': config.dataset_name,
        'learning_rate': LEARNING_RATE,
        'batch_size': BATCH_SIZE,
        'num_epochs': NUM_EPOCHS,
    }
}, model_path)
print(f"Model saved to: {model_path}")


Model saved to: ./VGG_cifar10_trained.pth
